In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('Data/UpdatedEstData.csv')
df.columns = [c.strip().upper() for c in df.columns]
df_obs = df[(df['EVID'] == 0) & (df['MDV'] == 0)].copy()

print(df_obs[['ID','DVID','TIME','DV','DOSE','BW']].head(10))

In [ ]:
PATIENT_IDS = [9, 13, 26, 46]

fig, axes = plt.subplots(nrows=len(PATIENT_IDS), ncols=2,
                         figsize=(12, 3.5 * len(PATIENT_IDS)))

for row, pid in enumerate(PATIENT_IDS):
    pat  = df_obs[df_obs['ID'] == pid].sort_values('TIME')
    pk   = pat[pat['DVID'] == 1]
    pd_  = pat[pat['DVID'] == 2]
    dose = pat['DOSE'].iloc[0] if len(pat) > 0 else 0
    bw   = pat['BW'].iloc[0]   if len(pat) > 0 else float('nan')

    # ── Left column: PK ──────────────────────────────────────
    ax_pk = axes[row, 0]
    if len(pk) > 0:
        ax_pk.plot(pk['TIME'], pk['DV'], 'o-', color='steelblue',
                   linewidth=1.8, markersize=5)
        ax_pk.set_ylabel('PK Concentration', fontsize=9)
    else:
        ax_pk.text(0.5, 0.5, 'No PK data\n(Placebo)',
                   ha='center', va='center', transform=ax_pk.transAxes,
                   color='gray', fontsize=11)
        ax_pk.set_ylabel('PK Concentration', fontsize=9)

    ax_pk.set_title(f'Patient {pid} — PK  (Dose={dose}, BW={bw:.1f} kg)', fontsize=10)
    ax_pk.set_xlabel('Time (hours)', fontsize=9)
    ax_pk.grid(True, alpha=0.3)

    # ── Right column: PD ─────────────────────────────────────
    ax_pd = axes[row, 1]
    if len(pd_) > 0:
        ax_pd.plot(pd_['TIME'], pd_['DV'], 's-', color='darkorange',
                   linewidth=1.8, markersize=5)
        ax_pd.set_ylabel('PD Biomarker', fontsize=9)
    else:
        ax_pd.text(0.5, 0.5, 'No PD data',
                   ha='center', va='center', transform=ax_pd.transAxes,
                   color='gray', fontsize=11)

    ax_pd.set_title(f'Patient {pid} — PD  (Dose={dose}, BW={bw:.1f} kg)', fontsize=10)
    ax_pd.set_xlabel('Time (hours)', fontsize=9)
    ax_pd.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualize_10Apr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visualize_10Apr.png')

In [ ]:
# Summary table
rows = []
for pid in PATIENT_IDS:
    pat  = df_obs[df_obs['ID'] == pid]
    pk   = pat[pat['DVID'] == 1]['DV']
    pd_  = pat[pat['DVID'] == 2]['DV']
    dose = pat['DOSE'].iloc[0] if len(pat) > 0 else 0
    bw   = pat['BW'].iloc[0]   if len(pat) > 0 else float('nan')
    rows.append({
        'Patient': pid, 'Dose': dose, 'BW': round(bw, 1),
        'PK obs': len(pk),
        'PK min': round(pk.min(), 3) if len(pk) else None,
        'PK max': round(pk.max(), 3) if len(pk) else None,
        'PK mean': round(pk.mean(), 3) if len(pk) else None,
        'PD obs': len(pd_),
        'PD min': round(pd_.min(), 3) if len(pd_) else None,
        'PD max': round(pd_.max(), 3) if len(pd_) else None,
        'PD mean': round(pd_.mean(), 3) if len(pd_) else None,
    })

pd.DataFrame(rows).set_index('Patient')